# OpenHeat v0.6 — live forecast and calibration notebook

This notebook demonstrates the v0.6 workflow: sample forecast, live forecast hook, NEA fixture parser, and calibration skeleton.


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
ROOT


In [ ]:
from openheat_forecast.live_pipeline import run_offline_sample_forecast
files = run_offline_sample_forecast(ROOT/'data/sample/openmeteo_heatwave_forecast_sample.csv', ROOT/'data/sample/toa_payoh_grid_sample.csv', out_dir=ROOT/'outputs')
files


In [ ]:
import pandas as pd
hotspots = pd.read_csv(ROOT/'outputs/v06_offline_hotspot_ranking.csv')
hotspots.head(10)


In [ ]:
from openheat_forecast.live_api import normalise_realtime_station_readings, merge_latest_station_observations, attach_nearest_station
fixtures = ROOT/'data/fixtures'
air = normalise_realtime_station_readings(json.loads((fixtures/'nea_air_temperature_sample.json').read_text()), 'air_temperature_c')
rh = normalise_realtime_station_readings(json.loads((fixtures/'nea_relative_humidity_sample.json').read_text()), 'relative_humidity_percent')
wind = normalise_realtime_station_readings(json.loads((fixtures/'nea_wind_speed_sample.json').read_text()), 'wind_speed_raw')
wind['wind_speed_ms'] = wind['wind_speed_raw'] * 0.514444
wbgt = normalise_realtime_station_readings(json.loads((fixtures/'nea_wbgt_sample.json').read_text()), 'official_wbgt_c')
obs = merge_latest_station_observations(air, rh, wind, wbgt)
obs


In [ ]:
grid = pd.read_csv(ROOT/'data/sample/toa_payoh_grid_sample.csv')
nearest = attach_nearest_station(grid.head(10), obs)
nearest[['cell_id','nearest_station_id','nearest_station_name','nearest_station_distance_m']].head()


In [ ]:
from openheat_forecast.calibration import fit_linear_calibration, apply_linear_calibration, station_skill_metrics
demo = obs[['station_id','official_wbgt_c']].dropna().copy()
demo['wbgt_proxy_c'] = demo['official_wbgt_c'] + [-0.7, 0.2, -0.3][:len(demo)]
model = fit_linear_calibration(demo)
corrected = apply_linear_calibration(demo, model)
model, station_skill_metrics(corrected, 'wbgt_calibrated_c', 'official_wbgt_c')


## Live mode

Run in a local environment with internet access:

```python
from openheat_forecast.live_pipeline import run_forecast_from_openmeteo, fetch_latest_nea_observation_bundle
run_forecast_from_openmeteo(ROOT/'data/sample/toa_payoh_grid_sample.csv', out_dir=ROOT/'outputs')
obs_live = fetch_latest_nea_observation_bundle()
```
